# Day 09. Exercise 00
# Regularization

## 0. Imports

In [70]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
import joblib as jb
import time

## 1. Preprocessing

In [71]:
df1 = pd.read_csv('../data/dayofweek.csv')
X = df1.drop(columns='dayofweek')
y = df1['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=21,
    stratify=y
)
df1.head()

,numTrials,hour,dayofweek,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,-0.788667,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.756764,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.724861,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.692958,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.661055,-2.562352,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 2. Logreg regularization

### a. Default regularization

In [72]:
def kfold(model, X=X_train, y=y_train, n=10):
    kf = StratifiedKFold(n_splits=n)
    accuracy = []

    for train_index, valid_index in kf.split(X, y):
        y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
        X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]

        model.fit(X_train, y_train)

        valid_pred = model.predict(X_valid)
        train_pred = model.predict(X_train)

        accuracy_valid = accuracy_score(y_valid, valid_pred)
        accuracy_train = accuracy_score(y_train, train_pred)
        print(f'train -  {accuracy_train:.5f}   |   valid -  {accuracy_valid:.5f}')

        accuracy.append(accuracy_valid)
    accuracy_res = np.mean(accuracy)
    std = np.std(accuracy)
    print(f'Average accuracy on crosvaal is {accuracy_res:.5f}')
    print(f'Std is {std:.5f}')
    return accuracy_res, std

In [73]:
%%time
logreg = LogisticRegression(random_state=21, fit_intercept=False)
kfold(logreg)

train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crosvaal is 0.60165
Std is 0.02943
CPU times: user 2.67 s, sys: 6.24 s, total: 8.91 s
Wall time: 462 ms


(0.6016473189607519, 0.029427270750289624)

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [74]:
parameters = [
    {'penalty': 'l1', 'solver': 'saga', 'name': 'L1 (Saga) Regularization'},
    {'penalty': 'l2', 'solver': 'saga', 'name': 'L2 (Saga) Regularization'},
    {'penalty': 'l2', 'solver': 'lbfgs', 'name': 'L2 (Lbfgs) Regularization'},
    {'penalty': 'l2', 'solver': 'newton-cg', 'name': 'L2 (Newton-cg) Regularization'},
    {'penalty': 'none', 'solver': 'saga', 'name': 'Non Regularization (Saga)'},
    {'penalty': 'none', 'solver': 'lbfgs', 'name': 'Non Regularization (Lbfgs)'},
    {'penalty': 'none', 'solver': 'newton-cg', 'name': 'Non Regularization (Newton-cg)'},
]

In [75]:
optimized_accur = {}
optimized_std = {}

for parameter in parameters:
    start_time = time.time()
    params = {k: i for k,i in parameter.items() if k not in 'name'}
    logreg = LogisticRegression(**params, random_state=21, fit_intercept=False, max_iter=5000)
    print(parameter['name'])
    accuracy, std = kfold(logreg)
    end_time = time.time()
    print(f"Time for {parameter['name']}: {end_time - start_time:.2f} seconds")
    print('\n')
    optimized_accur[parameter['name']] = accuracy
    optimized_std[parameter['name']] = std

L1 (Saga) Regularization
train -  0.63726   |   valid -  0.58519
train -  0.64221   |   valid -  0.61481
train -  0.62984   |   valid -  0.55556
train -  0.64386   |   valid -  0.60000
train -  0.63232   |   valid -  0.57778
train -  0.63644   |   valid -  0.57778
train -  0.63644   |   valid -  0.65926
train -  0.65622   |   valid -  0.57778
train -  0.64580   |   valid -  0.58955
train -  0.63839   |   valid -  0.62687
Average accuracy on crosvaal is 0.59646
Std is 0.02848
Time for L1 (Saga) Regularization: 2.76 seconds


L2 (Saga) Regularization
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crosvaal is 0.60165
Std i

In [76]:
log_res = pd.DataFrame({
    'accuracy': optimized_accur.values(),
    'std': optimized_std.values()
}, index=optimized_accur.keys())
log_res

,accuracy,std
L1 (Saga) Regularization,0.596457,0.028483
L2 (Saga) Regularization,0.601647,0.029427
L2 (Lbfgs) Regularization,0.601647,0.029427
L2 (Newton-cg) Regularization,0.601647,0.029427
Non Regularization (Saga),0.624616,0.033790
Non Regularization (Lbfgs),0.624616,0.033790
Non Regularization (Newton-cg),0.624616,0.033790


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [77]:
%%time
svc = SVC(kernel='linear', random_state=21, probability=True)
kfold(svc)

train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556
train -  0.69415   |   valid -  0.62222
train -  0.70239   |   valid -  0.65185
train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444
train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222
train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crosvaal is 0.65871
Std is 0.04359
CPU times: user 1.86 s, sys: 716 ms, total: 2.58 s
Wall time: 1.64 s


(0.6587064676616916, 0.043585708770590564)

### b. Optimizing regularization parameters

In [78]:
optimized_accur = {}
optimized_std = {}
for param in range(36, 46):
    start_time = time.time()
    svc = SVC(kernel='linear', random_state=21, probability=True, C=param)
    print(f'For C=={param}')
    accuracy, std = kfold(svc)
    end_time = time.time()
    print(f"Time for {parameter['name']}: {end_time - start_time:.2f} seconds")
    print('\n')
    optimized_accur[param] = accuracy
    optimized_std[param] = std

For C==36
train -  0.77906   |   valid -  0.74815
train -  0.79472   |   valid -  0.82963
train -  0.80462   |   valid -  0.71852
train -  0.77659   |   valid -  0.75556
train -  0.78071   |   valid -  0.77037
train -  0.79472   |   valid -  0.74074
train -  0.78318   |   valid -  0.77037
train -  0.79885   |   valid -  0.72593
train -  0.79407   |   valid -  0.70896
train -  0.79984   |   valid -  0.73881
Average accuracy on crosvaal is 0.75070
Std is 0.03266
Time for Non Regularization (Newton-cg): 4.39 seconds


For C==37
train -  0.77988   |   valid -  0.74815
train -  0.79472   |   valid -  0.82963
train -  0.80379   |   valid -  0.71852
train -  0.77659   |   valid -  0.76296
train -  0.78071   |   valid -  0.77778
train -  0.79555   |   valid -  0.74074
train -  0.78318   |   valid -  0.77037
train -  0.79802   |   valid -  0.72593
train -  0.79407   |   valid -  0.70896
train -  0.79984   |   valid -  0.73881
Average accuracy on crosvaal is 0.75218
Std is 0.03334
Time for Non R

In [79]:
svm_res = pd.DataFrame({
    'accuracy': optimized_accur.values(),
    'std': optimized_std.values()
}, index=optimized_accur.keys())
svm_res

,accuracy,std
36,0.750702,0.032660
37,0.752184,0.033343
38,0.750702,0.032491
39,0.751443,0.033011
40,0.752924,0.032662
41,0.751443,0.033011
42,0.751443,0.031827
43,0.752924,0.032662
44,0.752184,0.033343
45,0.752924,0.033327


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [80]:
%%time
tree = DecisionTreeClassifier(max_depth=10, random_state=21)
kfold(tree)

train -  0.81039   |   valid -  0.74074
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.76296
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.75373
train -  0.80478   |   valid -  0.68657
Average accuracy on crosvaal is 0.72551
Std is 0.03562
CPU times: user 38.1 ms, sys: 0 ns, total: 38.1 ms
Wall time: 38.3 ms


(0.7255113322277501, 0.03562429627613885)

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [81]:
optimized_accur = {}
optimized_std = {}
for n in range(1,22,2):
    start_time = time.time()
    tree = DecisionTreeClassifier(max_depth=n, random_state=21)
    print(f'\nFor max_depth=={n}')
    accuracy, std = kfold(tree)
    end_time = time.time()
    print(f"Time for {parameter['name']}: {end_time - start_time:.2f} seconds")
    optimized_accur[n] = accuracy
    optimized_std[n] = std


For max_depth==1
train -  0.35367   |   valid -  0.37037
train -  0.35449   |   valid -  0.36296
train -  0.35614   |   valid -  0.34815
train -  0.35449   |   valid -  0.36296
train -  0.35532   |   valid -  0.35556
train -  0.35367   |   valid -  0.37037
train -  0.35532   |   valid -  0.35556
train -  0.35614   |   valid -  0.34815
train -  0.35667   |   valid -  0.34328
train -  0.35750   |   valid -  0.33582
Average accuracy on crosvaal is 0.35532
Std is 0.01094
Time for Non Regularization (Newton-cg): 0.03 seconds

For max_depth==3
train -  0.49382   |   valid -  0.43704
train -  0.48887   |   valid -  0.48148
train -  0.50206   |   valid -  0.44444
train -  0.49629   |   valid -  0.49630
train -  0.48475   |   valid -  0.48889
train -  0.48969   |   valid -  0.48889
train -  0.48392   |   valid -  0.48148
train -  0.49052   |   valid -  0.40741
train -  0.48517   |   valid -  0.46269
train -  0.49176   |   valid -  0.42537
Average accuracy on crosvaal is 0.46140
Std is 0.02938


In [82]:
tree_res = pd.DataFrame({
    'max_depth': optimized_accur.keys(),
    'accuracy': optimized_accur.values(),
    'std': optimized_std.values()
})
tree_res

,max_depth,accuracy,std
0,1,0.355318,0.010945
1,3,0.461399,0.029379
2,5,0.543013,0.024234
3,7,0.649889,0.039706
4,9,0.703250,0.040683
5,11,0.769989,0.037899
6,13,0.830088,0.023767
7,15,0.854588,0.026816
8,17,0.873875,0.022811
9,19,0.880542,0.021732


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [83]:
%%time
forest = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
kfold(forest)

train -  0.96455   |   valid -  0.88148
train -  0.96208   |   valid -  0.91852
train -  0.96785   |   valid -  0.86667
train -  0.96455   |   valid -  0.89630
train -  0.96538   |   valid -  0.91111
train -  0.96538   |   valid -  0.88148
train -  0.97115   |   valid -  0.91852
train -  0.96867   |   valid -  0.85185
train -  0.97364   |   valid -  0.88060
train -  0.97941   |   valid -  0.86567
Average accuracy on crosvaal is 0.88722
Std is 0.02204
CPU times: user 436 ms, sys: 0 ns, total: 436 ms
Wall time: 445 ms


(0.8872194582642343, 0.022044865936245422)

### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [84]:
res = []
for j in range(15,25,5):
    for i in range(100,150,10):
        start_time = time.time()
        tree = RandomForestClassifier(n_estimators=i, max_depth=j, random_state=21)
        print(f'\nFor n_estimators=={i}')
        print(f'For max_depth=={j}')
        accuracy, std = kfold(tree)
        end_time = time.time()
        print(f"Time for {parameter['name']}: {end_time - start_time:.2f} seconds")
        res.append({
            'max_depth': j,
            'n__estimators': i,
            'accuracy': accuracy,
            'std': std
        })
fores_res = pd.DataFrame(res)
fores_res


For n_estimators==100
For max_depth==15
train -  0.97774   |   valid -  0.89630
train -  0.97939   |   valid -  0.93333
train -  0.98269   |   valid -  0.88148
train -  0.98186   |   valid -  0.90370
train -  0.98021   |   valid -  0.90370
train -  0.98186   |   valid -  0.88148
train -  0.98599   |   valid -  0.91852
train -  0.98186   |   valid -  0.87407
train -  0.98023   |   valid -  0.88806
train -  0.98023   |   valid -  0.88060
Average accuracy on crosvaal is 0.89612
Std is 0.01795
Time for Non Regularization (Newton-cg): 0.87 seconds

For n_estimators==110
For max_depth==15
train -  0.97939   |   valid -  0.90370
train -  0.98104   |   valid -  0.93333
train -  0.98186   |   valid -  0.88148
train -  0.98269   |   valid -  0.90370
train -  0.98186   |   valid -  0.90370
train -  0.98516   |   valid -  0.87407
train -  0.98516   |   valid -  0.91852
train -  0.98186   |   valid -  0.87407
train -  0.98270   |   valid -  0.88806
train -  0.98188   |   valid -  0.88060
Average a

,max_depth,n__estimators,accuracy,std
0,15,100,0.896125,0.017950
1,15,110,0.896125,0.018845
2,15,120,0.898347,0.020008
3,15,130,0.896866,0.017477
4,15,140,0.898353,0.017344
5,20,100,0.910227,0.019250
6,20,110,0.907999,0.021855
7,20,120,0.907999,0.020826
8,20,130,0.908740,0.019428
9,20,140,0.910967,0.022058


In [85]:
fores_res.nlargest(3, 'accuracy')

,max_depth,n__estimators,accuracy,std
9,20,140,0.910967,0.022058
5,20,100,0.910227,0.019250
8,20,130,0.908740,0.019428


In [86]:
tree_res.nlargest(3, 'accuracy')

,max_depth,accuracy,std
10,21,0.882775,0.017570
9,19,0.880542,0.021732
8,17,0.873875,0.022811


In [87]:
svm_res.nlargest(3, 'accuracy')

,accuracy,std
40,0.752924,0.032662
43,0.752924,0.032662
45,0.752924,0.033327


In [88]:
log_res.nlargest(3, 'accuracy')

,accuracy,std
Non Regularization (Saga),0.624616,0.03379
Non Regularization (Lbfgs),0.624616,0.03379
Non Regularization (Newton-cg),0.624616,0.03379


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [89]:
model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=21).fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.9319526627218935

In [90]:
cm = confusion_matrix(y_test, y_pred)
res = []
for day in range(0,7):
    total =  np.sum(y_test==day)
    correct = cm[day, day]
    error = total - correct
    percent = round((error / total) * 100, 2)
    res.append({
        'total': total,
        'correct': correct,
        'error': error,
        'percent': percent
    })
pd.DataFrame(res)

,total,correct,error,percent
0,27,20,7,25.93
1,55,50,5,9.09
2,30,28,2,6.67
3,80,78,2,2.50
4,21,18,3,14.29
5,54,51,3,5.56
6,71,70,1,1.41


In [91]:
jb.dump(model, 'best_mod.jolib')

['best_mod.jolib']